In [13]:
import sagemaker
from sagemaker.session import Session
from sagemaker.workflow.pipeline_context import PipelineSession

region = "us-west-2"
default_bucket = "sm-churn-7"
input_data = f"s3://{default_bucket}/data/storedata_total.csv"  
batch_data = input_data  

sess = sagemaker.Session()
p_sess = PipelineSession()
role = sagemaker.get_execution_role()

processing_instance_type = "ml.t3.medium"  
processing_instance_count = 1
training_instance_type   = "ml.m5.large"

print(region, default_bucket, role)


us-west-2 sm-churn-7 arn:aws:iam::255550682979:role/service-role/AmazonSageMaker-ExecutionRole-20251111T001596


In [7]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

step_process = ProcessingStep(
    name="ChurnModelProcess",
    processor=SKLearnProcessor(
        framework_version="0.23-1",
        instance_type=processing_instance_type,
        instance_count=processing_instance_count,
        role=role,
        sagemaker_session=p_sess,
    ),
    inputs=[
        ProcessingInput(source=input_data, destination="/opt/ml/processing/input"),
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/train",      destination=f"s3://{default_bucket}/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation", destination=f"s3://{default_bucket}/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/test",       destination=f"s3://{default_bucket}/output/test"),
    ],
    code=f"s3://{default_bucket}/input/code/preprocess.py",
)
step_process


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


In [14]:
from sagemaker.image_uris import retrieve as get_image_uri
from sagemaker.estimator import Estimator          
from sagemaker.inputs import TrainingInput         
from sagemaker.tuner import ContinuousParameter, IntegerParameter, HyperparameterTuner
from sagemaker.workflow.steps import TuningStep

image_uri = get_image_uri(
    framework="xgboost",
    region=region,
    version="1.7-1",      
    py_version="py3",
    instance_type=training_instance_type,
)

fixed_hps = {
    "eval_metric": "auc",
    "objective": "binary:logistic",
    "num_round": "100",
    "rate_drop": "0.3",
    "tweedie_variance_power": "1.4",
}

xgb_train = Estimator(
    image_uri=image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    hyperparameters=fixed_hps,
    output_path=f"s3://{default_bucket}/output",
    base_job_name="churn-train",
    sagemaker_session=p_sess,
    role=role,
)

hp_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}

tuner = HyperparameterTuner(
    estimator=xgb_train,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hp_ranges,
    max_jobs=4,
    max_parallel_jobs=1,   
)

step_tuning = TuningStep(
    name="ChurnHyperParameterTuning",
    tuner=tuner,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)
step_tuning


INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.


In [24]:
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ProcessingInput as PIn, ProcessingOutput as POut

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

eval_processor = SKLearnProcessor(
    framework_version="0.23-1",
    instance_type=processing_instance_type,
    instance_count=1,
    role=role,
    sagemaker_session=p_sess,
)

step_eval = ProcessingStep(
    name="ChurnEvalBestModel",
    processor=eval_processor,
    inputs=[
        PIn(
            source=step_tuning.get_top_model_s3_uri(top_k=0, s3_bucket=default_bucket, prefix="output"),
            destination="/opt/ml/processing/model",
        ),
        PIn(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        POut(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code=f"s3://{default_bucket}/input/code/evaluate.py",
    property_files=[evaluation_report],
)
step_eval


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


In [ ]:
# --- 5) Register best model to Model Registry ---
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.functions import Join

# Join S3 path for evaluation.json
eval_json_s3 = Join(
    on="/",
    values=[
        step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
        "evaluation.json",
    ],
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=eval_json_s3,
        content_type="application/json",
    )
)

step_register = RegisterModel(
    name="RegisterChurnModel",
    estimator=xgb_train,
    model_data=step_tuning.get_top_model_s3_uri(top_k=0, s3_bucket=default_bucket, prefix="output"),
    content_types=["text/csv"],
    response_types=["text/csv"],
    # Declares inference/batch inference capability used by the model package
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="ChurnModelPackageGroup",
    model_metrics=model_metrics,
)

# --- 6) Create Model ---
from sagemaker.workflow.model_step import ModelStep
from sagemaker.model import Model as SMModel

create_model = SMModel(
    image_uri=image_uri,
    model_data=step_tuning.get_top_model_s3_uri(top_k=0, s3_bucket=default_bucket, prefix="output"),
    role=role,
    sagemaker_session=p_sess,
)
create_model_step_args = create_model.create()

step_create_model = ModelStep(
    name="ChurnCreateModel",
    step_args=create_model_step_args,
)

# --- 7) Condition: register/create model only if AUC > 0.75 ---
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.condition_step import ConditionStep

cond_auc_ok = ConditionGreaterThan(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,   # PropertyFile(name="EvaluationReport", path="evaluation.json")
        json_path="classification_metrics.auc_score.value",
    ),
    right=0.75,
)

step_cond = ConditionStep(
    name="CheckAUCScoreChurnEvaluation",
    conditions=[cond_auc_ok],
    if_steps=[step_register, step_create_model],  # Register & create model only if condition satisfied
    else_steps=[],
)


In [ ]:
print("step_eval.name =", step_eval.name, "| type:", type(step_eval.name))

from sagemaker.workflow.properties import PropertyFile

print("evaluation_report type:", type(evaluation_report))

assert isinstance(step_eval.name, str), "step_eval.name is not a string!"
assert isinstance(evaluation_report, PropertyFile), "evaluation_report is not a PropertyFile!"


step_eval.name = ChurnEvalBestModel | type: <class 'str'>
evaluation_report type: <class 'sagemaker.workflow.properties.PropertyFile'>


In [ ]:
from sagemaker.workflow.pipeline import Pipeline
import time

# Assemble pipeline
# pipeline = Pipeline(
#     name="ChurnPipeline",
#     steps=[step_process, step_tuning, step_eval, step_cond],
#     sagemaker_session=p_sess,
# )
pipeline = Pipeline(
    name="ChurnPipeline",
    steps=[
        step_process,
        step_tuning,
        step_eval,
        step_cond   # Condition contains register/create/transform/clarify
    ],
    sagemaker_session=p_sess,
)

# Create / Update
pipeline.upsert(role_arn=role)

# Start execution
execution = pipeline.start()
info = execution.describe()
print("Started:", info["PipelineExecutionArn"], "| Status:", info["PipelineExecutionStatus"])

# Poll until terminal state (Succeeded/Failed/Stopped), print step status every 30 sec
terminal = {"Succeeded", "Failed", "Stopped"}
last_print = None

while True:
    status = execution.describe()["PipelineExecutionStatus"]
    steps = execution.list_steps()

    # Print only when state changes or first time
    snapshot = [(s["StepName"], s["StepStatus"]) for s in steps]
    if snapshot != last_print:
        print("Pipeline status:", status)
        for s in steps:
            # FIXED TYPO: "SatepName" → "StepName"
            print(f" - {s['StepName']}: {s['StepStatus']}",
                  ("| Reason: " + s.get("FailureReason", "")) if s["StepStatus"] == "Failed" else "")
        last_print = snapshot

    if status in terminal:
        break

    time.sleep(30)

print("Final pipeline status:", status)

# If failed, summarize failed steps and raise a clear error
if status == "Failed":
    failed = [s for s in steps if s["StepStatus"] == "Failed"]
    print("\nFailed steps summary:")
    for s in failed:
        print(f" * {s['StepName']}: {s.get('FailureReason')}")
    # raise RuntimeError("Pipeline failed; see failed steps above.")


Started: arn:aws:sagemaker:us-west-2:255550682979:pipeline/ChurnPipeline/execution/myhqs3b19cld | Status: Executing
Pipeline status: Executing
Pipeline status: Executing
 - ChurnModelProcess: Executing 
Pipeline status: Executing
 - ChurnHyperParameterTuning: Executing 
 - ChurnModelProcess: Succeeded 
Pipeline status: Executing
 - ChurnEvalBestModel: Executing 
 - ChurnHyperParameterTuning: Succeeded 
 - ChurnModelProcess: Succeeded 
Pipeline status: Succeeded
 - RegisterChurnModel-RegisterModel: Succeeded 
 - ChurnCreateModel-CreateModel: Succeeded 
 - CheckAUCScoreChurnEvaluation: Succeeded 
 - ChurnEvalBestModel: Succeeded 
 - ChurnHyperParameterTuning: Succeeded 
 - ChurnModelProcess: Succeeded 
Final pipeline status: Succeeded
